# Unauthorized physical access (tailgating, piggybacking, lock bypass)  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 2.2.a: Detect unauthorized physical access in a badge log

**Skill:** Analyze Risk · **Maps to:** 2.2.A, 2.2.B · **Language:** Python 3 (pytest)

## Problem
A building security system writes one log record per door event. Your job is to implement the
analyst's first-pass filter: given a list of log records, return the event IDs of all records
that indicate a likely unauthorized physical access attempt, based on two detection rules:

**Rule 1, Impossible speed (tailgating / badge pass-back indicator).**
If the same badge ID appears at two different doors and the time between events is less than
the minimum walk time (distance_meters / MAX_WALK_SPEED), the second event is suspicious.
`MAX_WALK_SPEED = 2.0` meters per second.

**Rule 2, Sensor mismatch (tailgating indicator).**
If an event has `"sensor_count": 2` (or higher) but only one badge swipe was recorded (i.e.,
the event itself has `"badge": <a badge id>` and no second badge event at the same door within
1 second), that event is suspicious.

Return the list of flagged event IDs (strings), sorted ascending. An event may match both rules
but should appear only once.

## Function contract
Implement `solve(records: list[dict]) -> list[str]` in the code editor
(see the Reference section below).

## Input
`records`, a list of dicts, each with keys:
- `"id"` (str): unique event identifier, e.g. `"E001"`
- `"time"` (int): Unix timestamp in seconds (integer)
- `"badge"` (str | None): badge ID that was swiped, or `None` if no swipe
- `"door"` (str): door identifier, e.g. `"D1"`
- `"distance_from_prev"` (int | None): meters from the previous door this badge was seen at,
  or `None` if this is the badge's first appearance or badge is None
- `"sensor_count"` (int): number of bodies the door sensor detected crossing (1 = normal)

Records are provided in chronological order (ascending `"time"`).

## Output
A sorted `list[str]` of flagged event IDs. Empty list if no violations found.

## Examples
```
[
  {"id":"E001","time":1000,"badge":"B-204","door":"D1","distance_from_prev":None,"sensor_count":1},
  {"id":"E002","time":1003,"badge":"B-204","door":"D2","distance_from_prev":50,"sensor_count":1},
  {"id":"E003","time":2000,"badge":"B-117","door":"D1","distance_from_prev":None,"sensor_count":2},
]
-> ["E002", "E003"]

E002: B-204 covers 50 m in 3 s = 16.7 m/s > 2.0 m/s  (Rule 1)
E003: sensor_count=2 with one swipe                    (Rule 2)
```

## Constraints
- Deterministic, offline, standard library only.
- MAX_WALK_SPEED = 2.0 m/s (defined in the Reference section below).
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(records: list[dict]) -> list[str]
```

### Input (`records`)
A list of dicts in chronological order, each with:
- `"id"` (str): unique event ID, e.g. `"E001"`
- `"time"` (int): Unix timestamp in whole seconds
- `"badge"` (str | None): badge ID swiped, or None if no swipe
- `"door"` (str): door identifier
- `"distance_from_prev"` (int | None): distance in meters from where this badge was last seen,
  or None if badge is None or first appearance
- `"sensor_count"` (int): number of bodies detected crossing the threshold (normally 1)

### Output (`result`)
A **sorted, de-duplicated** `list[str]` of flagged event IDs. Empty list if none.

### Detection rules
- **Rule 1, Impossible speed:** A record is flagged if `badge` is not None,
  `distance_from_prev` is not None, and
  `distance_from_prev / (time - prev_time_for_this_badge) > MAX_WALK_SPEED`.
  `MAX_WALK_SPEED = 2.0` (meters per second).
  Division by zero (same timestamp): always flag.
- **Rule 2, Sensor mismatch:** A record is flagged if `sensor_count >= 2` and
  `badge` is not None (a door with no swipe at all is a separate alarm; not in scope).

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Records are always in ascending time order.
- Output is sorted ascending and de-duplicated (an event matching both rules appears once).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 2.2.a. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import List, Dict, Any

MAX_WALK_SPEED = 2.0  # meters per second


def solve(records: List[Dict[str, Any]]) -> List[str]:
    """Return sorted list of event IDs that indicate unauthorized physical access.

    Rule 1 — Impossible speed: badge appears at a new door faster than MAX_WALK_SPEED.
    Rule 2 — Sensor mismatch: sensor_count >= 2 with a single badge swipe.
    """
    flagged = set()
    # Track last seen time per badge for Rule 1
    last_time: Dict[str, int] = {}

    for rec in records:
        eid = rec["id"]
        badge = rec["badge"]
        t = rec["time"]
        dist = rec["distance_from_prev"]
        sensor = rec["sensor_count"]

        # Rule 1: impossible speed
        if badge is not None and dist is not None:
            if badge in last_time:
                elapsed = t - last_time[badge]
                if elapsed == 0 or dist / elapsed > MAX_WALK_SPEED:
                    flagged.add(eid)
            last_time[badge] = t
        elif badge is not None:
            last_time[badge] = t

        # Rule 2: sensor mismatch (two or more bodies, only one badge)
        if badge is not None and sensor >= 2:
            flagged.add(eid)

    return sorted(flagged)


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- helpers ---------------------------------------------------------------
def mk(eid, time, badge, door, dist, sensor):
    return {"id": eid, "time": time, "badge": badge, "door": door,
            "distance_from_prev": dist, "sensor_count": sensor}


# ---- visible cases ---------------------------------------------------------
def test_impossible_speed_flagged():
    # B-204 covers 50 m in 3 s = 16.7 m/s > 2.0 — flag E002
    records = [
        mk("E001", 1000, "B-204", "D1", None, 1),
        mk("E002", 1003, "B-204", "D2", 50,   1),
    ]
    assert impl.solve(records) == ["E002"]


def test_sensor_mismatch_flagged():
    # sensor_count=2 with one badge swipe — flag E003
    records = [
        mk("E003", 2000, "B-117", "D1", None, 2),
    ]
    assert impl.solve(records) == ["E003"]


def test_clean_entry_not_flagged():
    # 10 m in 10 s = 1.0 m/s <= 2.0; sensor=1 — clean
    records = [
        mk("E004", 3000, "B-011", "D1", None, 1),
        mk("E005", 3010, "B-011", "D2", 10,   1),
    ]
    assert impl.solve(records) == []


def test_empty_log():
    assert impl.solve([]) == []


def test_both_rules_same_event():
    # sensor=2 AND speed violation on same record — appears once
    records = [
        mk("E010", 5000, "B-300", "D1", None, 1),
        mk("E011", 5001, "B-300", "D2", 30,   2),  # 30/1=30 m/s AND sensor=2
    ]
    result = impl.solve(records)
    assert result == ["E011"]
    assert result.count("E011") == 1


def test_multiple_badges_only_one_flagged():
    # B-001 is clean; B-002 has impossible speed
    records = [
        mk("E020", 6000, "B-001", "D1", None, 1),
        mk("E021", 6100, "B-001", "D2", 100,  1),   # 100/100=1.0 m/s — OK
        mk("E022", 6000, "B-002", "D1", None, 1),
        mk("E023", 6002, "B-002", "D2", 20,   1),   # 20/2=10 m/s — flagged
    ]
    assert impl.solve(records) == ["E023"]


# ---- hidden / edge cases ---------------------------------------------------
def test_same_timestamp_flagged():
    # elapsed=0 — always flag (division by zero case)
    records = [
        mk("E030", 7000, "B-400", "D1", None, 1),
        mk("E031", 7000, "B-400", "D2", 1,    1),
    ]
    assert impl.solve(records) == ["E031"]


def test_no_badge_not_flagged_by_speed():
    # badge=None rows should not trigger Rule 1 or Rule 2
    records = [
        mk("E040", 8000, None, "D1", None, 1),
        mk("E041", 8001, None, "D1", 5,    1),
    ]
    assert impl.solve(records) == []


def test_exactly_at_speed_limit_not_flagged():
    # 20 m in 10 s = exactly 2.0 m/s — not strictly greater than limit
    records = [
        mk("E050", 9000, "B-500", "D1", None, 1),
        mk("E051", 9010, "B-500", "D2", 20,   1),
    ]
    assert impl.solve(records) == []


def test_sorted_output():
    # Multiple violations — result must be sorted
    records = [
        mk("E060", 10000, "B-600", "D1", None, 1),
        mk("E061", 10001, "B-600", "D2", 50,   2),  # both rules
        mk("E062", 10100, "B-700", "D1", None, 2),  # Rule 2
    ]
    result = impl.solve(records)
    assert result == sorted(result)
    assert "E061" in result
    assert "E062" in result


def test_first_appearance_no_flag():
    # First time a badge is seen — no distance_from_prev, no speed check
    records = [
        mk("E070", 11000, "B-800", "D1", None, 1),
    ]
    assert impl.solve(records) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
